# clickstream-pipeline -- walkthrough

This notebook drives the **real** pure-Python core of `clickstream-pipeline`: no Kafka, no Spark, only numpy / pandas + the standard library. It runs the seeded demo and then demonstrates the bounded-memory and event-time primitives, each with the small hand-derived example pinned by the test suite.

Everything here is deterministic and reproducible.

In [ ]:
import clickstream
from clickstream import (
    funnel_time_to_convert,
    reorder_within_lateness,
    retention,
    top_k_heavy_hitters,
)
from clickstream.demo import run_demo

print('clickstream', clickstream.__version__)

## 1. The seeded demo

`run_demo(0)` generates a small synthetic clickstream (200 users, 659 events) and drives the real windowing, sessionisation, and funnel core over it. The numbers are pinned by a test and quoted in the README.

In [ ]:
metrics = run_demo(0)
print('total events :', metrics['total_events'])
print('num users    :', metrics['num_users'])
print('num sessions :', metrics['num_sessions'])
print('peak ev/min  :', metrics['peak_events_per_min'])
for s in metrics['funnel_steps']:
    print(f"  {s['step']:>12}: {s['users']:>3}  "
          f"(conv from top {s['conversion_from_top']:.2%})")

## 2. Heavy hitters (Misra-Gries)

`top_k_heavy_hitters(keys, k)` finds the most frequent event keys with a bounded-counter summary, then reports exact counts via a second pass. With two counters on the stream below the survivors are `{a, c}` and the exact top-2 is `[(a, 4), (c, 3)]`.

In [ ]:
keys = ['a', 'a', 'a', 'b', 'b', 'c', 'a', 'c', 'c']
print('with 2 counters:', top_k_heavy_hitters(keys, 2, counters=2))
print('default capacity:', top_k_heavy_hitters(keys, 2))

## 3. Out-of-order reordering within allowed lateness

`reorder_within_lateness(events, allowed_lateness_s)` buffers an out-of-order stream and emits it in timestamp order, dropping events that fall behind the watermark (`max_ts - allowed_lateness`). Here the event at ts=9 is too late and is dropped; the rest come out sorted.

In [ ]:
events = [(10.0, 'a'), (12.0, 'b'), (11.0, 'c'), (9.0, 'd'), (13.0, 'e')]
ordered, dropped = reorder_within_lateness(events, 2.0)
print('ordered:', ordered)
print('dropped too late:', dropped)

## 4. Funnel time-to-convert

`funnel_time_to_convert(user_events, steps)` returns the median seconds between consecutive completed funnel steps. For `view -> cart -> buy` the medians work out to `[10.0, 32.5]`.

In [ ]:
user_events = {
    'u1': [(0.0, 'view'), (10.0, 'cart'), (30.0, 'buy')],
    'u2': [(0.0, 'view'), (20.0, 'cart')],
    'u3': [(5.0, 'view'), (5.0, 'cart'), (50.0, 'buy')],
}
print(funnel_time_to_convert(user_events, ['view', 'cart', 'buy']))

## 5. Retention across periods

`retention(user_events_with_ts, period_s)` reports, for each pair of consecutive periods, the fraction of the earlier period's users who returned. With `period_s=100` the transitions resolve to `2/3` then `1/2`.

In [ ]:
users = {
    'u1': [10.0, 120.0],
    'u2': [50.0, 150.0, 250.0],
    'u3': [30.0],
    'u4': [220.0],
}
print(retention(users, 100.0))

### What these numbers do not prove

These are point answers over a tiny, seeded stream. They describe the stream; they do not explain it. The funnel and retention figures are not causal, and they depend on the chosen window, watermark, session gap, and period. See `README.md` and `USAGE.md` for the honest interpretation guide.